# Notebook 01 — Data Cleaning

**Project:** EU Funding Intelligence — EIF Financial Intermediaries  
**Source:** European Commission — Access to EU Finance Portal  
**Purpose:** Transform raw Excel data into a clean, structured CSV ready for analysis and Power BI

---

## What This Notebook Does
1. Loads the raw Excel file from the EU portal research
2. Assigns proper column names
3. Cleans text fields (whitespace, line breaks, encoding)
4. Extracts and standardizes key values (country, EIF amount, fund type)
5. Adds derived fields for analysis (SDG tags, project fit flags)
6. Exports a clean CSV to `01_data/processed/`

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 1. Load Raw Data

In [ ]:
raw_path = '../01_data/raw/eif_fund_managers.xlsx'

df_raw = pd.read_excel(raw_path, sheet_name='EIF', header=None)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

## 2. Assign Column Names

Row 1 of the Excel contains the field headers. We extract these as column names and drop the header rows.

In [ ]:
# Row index 1 contains headers (row 0 is blank)
column_names = [
    'location',
    'fund_manager',
    'financial_product',
    'target_area',
    'match_our_projects',
    'opportunities',
    'how_to_approach',
    'support_type',
    'eif_commitment',
    'address',
    'website',
    'email',
    'phone'
]

# Skip header rows (rows 0 and 1), use data from row 2 onwards
df = df_raw.iloc[2:].copy()
df.columns = column_names
df = df.reset_index(drop=True)

print(f'Data shape after removing headers: {df.shape}')
df.head()

## 3. Clean Text Fields

Raw text contains newlines, extra spaces, and special characters from copy-paste. We standardize all text fields.

In [ ]:
def clean_text(val):
    if pd.isna(val) or str(val).strip() in ['nan', 'None', '']:
        return None
    val = str(val)
    val = val.replace('\n', ' ').replace('\r', ' ')
    val = re.sub(r'\s+', ' ', val)
    return val.strip()

text_cols = ['location', 'fund_manager', 'financial_product', 'target_area',
             'opportunities', 'how_to_approach', 'support_type', 'address',
             'website', 'email', 'phone']

for col in text_cols:
    df[col] = df[col].apply(clean_text)

print('Text fields cleaned.')
df[['location', 'fund_manager', 'support_type']].head()

## 4. Extract and Standardize Country

The `location` field contains values like `Spain`, `Global`, `Spain & Portugal`, `Worldwide & Spain`. We extract a clean `country_primary` field.

In [ ]:
def extract_country(loc):
    if loc is None:
        return 'Unknown'
    loc_lower = loc.lower()
    if 'spain' in loc_lower and 'portugal' in loc_lower:
        return 'Spain & Portugal'
    elif 'spain' in loc_lower:
        return 'Spain'
    elif 'portugal' in loc_lower:
        return 'Portugal'
    elif 'sweden' in loc_lower or 'nordic' in loc_lower:
        return 'Sweden'
    elif 'global' in loc_lower or 'worldwide' in loc_lower:
        return 'Global'
    return loc

df['country_primary'] = df['location'].apply(extract_country)
print(df['country_primary'].value_counts())

## 5. Clean EIF Commitment Amount

The `eif_commitment` field contains values like `EUR 41,250,000` — we extract the numeric value.

In [ ]:
def extract_amount(val):
    if val is None:
        return None
    # Extract numeric value from strings like 'EUR 41,250,000'
    numbers = re.findall(r'[\d,\.]+', str(val))
    if numbers:
        # Take the largest number found (handles cases with multiple values)
        amounts = []
        for n in numbers:
            try:
                amounts.append(float(n.replace(',', '')))
            except:
                pass
        return max(amounts) if amounts else None
    return None

df['eif_commitment_eur'] = df['eif_commitment'].apply(extract_amount)
print('EIF commitment amounts extracted:')
print(df[['fund_manager', 'eif_commitment', 'eif_commitment_eur']].to_string())

## 6. Classify Fund Type

We derive a `fund_type` category from the `support_type` and `financial_product` fields.

In [ ]:
def classify_fund_type(row):
    text = ' '.join(filter(None, [
        str(row.get('support_type', '') or ''),
        str(row.get('financial_product', '') or ''),
        str(row.get('opportunities', '') or '')
    ])).lower()
    
    if 'accelerator' in text or 'sprint' in text:
        return 'Accelerator'
    elif 'infrastructure' in text:
        return 'Infrastructure'
    elif 'venture capital' in text or 'vc' in text or 'fcr' in text or 'scr' in text:
        return 'VC'
    elif 'debt' in text or 'loan' in text:
        return 'Debt'
    elif 'private equity' in text or 'generalist' in text:
        return 'PE'
    return 'Other'

df['fund_type'] = df.apply(classify_fund_type, axis=1)
print(df['fund_type'].value_counts())

## 7. Extract Fund Manager Name

The `fund_manager` field contains the name followed by a long description. We extract just the company name.

In [ ]:
def extract_manager_name(val):
    if val is None:
        return None
    # Take the first line or first sentence as the name
    name = val.split('.')[0].split(',')[0].strip()
    # Limit to reasonable length
    return name[:60] if len(name) > 60 else name

df['manager_name'] = df['fund_manager'].apply(extract_manager_name)
print(df[['manager_name', 'country_primary', 'fund_type']].to_string())

## 8. Add Project Fit Flags

We flag each fund manager for relevance to our three target project types: Solar, Hydrogen, and Desalination.

In [ ]:
def flag_project(row, keywords):
    text = ' '.join(filter(None, [
        str(row.get('how_to_approach', '') or ''),
        str(row.get('opportunities', '') or ''),
        str(row.get('target_area', '') or ''),
        str(row.get('financial_product', '') or '')
    ])).lower()
    return any(kw in text for kw in keywords)

df['solar_fit'] = df.apply(lambda r: flag_project(r, ['solar', 'photovoltaic', 'pv', 'renewable energy', 'energy transition']), axis=1)
df['hydrogen_fit'] = df.apply(lambda r: flag_project(r, ['hydrogen', 'electrolysis', 'decarboni']), axis=1)
df['desalination_fit'] = df.apply(lambda r: flag_project(r, ['desalination', 'water', 'sdg 6', 'sdg6', 'clean water', 'sanitation']), axis=1)

print('Project fit flags:')
print(df[['manager_name', 'solar_fit', 'hydrogen_fit', 'desalination_fit']].to_string())

## 9. Add SDG Tags

Extract SDG numbers mentioned in the how_to_approach and opportunities fields.

In [ ]:
def extract_sdgs(row):
    text = ' '.join(filter(None, [
        str(row.get('how_to_approach', '') or ''),
        str(row.get('target_area', '') or '')
    ]))
    # Find SDG numbers (SDG 6, SDG7, etc.)
    sdgs = re.findall(r'sdg\s*(\d+)', text, re.IGNORECASE)
    return sorted(list(set([int(s) for s in sdgs]))) if sdgs else []

df['sdg_tags'] = df.apply(extract_sdgs, axis=1)
df['sdg_count'] = df['sdg_tags'].apply(len)

print(df[['manager_name', 'sdg_tags']].to_string())

## 10. Final Clean Dataset

In [ ]:
# Select and reorder columns for the clean output
clean_cols = [
    'manager_name', 'country_primary', 'fund_type', 'support_type',
    'eif_commitment_eur', 'solar_fit', 'hydrogen_fit', 'desalination_fit',
    'sdg_count', 'sdg_tags', 'financial_product', 'target_area',
    'opportunities', 'how_to_approach', 'address', 'website', 'email', 'phone',
    'fund_manager', 'location', 'eif_commitment'
]

df_clean = df[clean_cols].copy()

# Convert boolean flags to integers for Power BI compatibility
df_clean['solar_fit'] = df_clean['solar_fit'].astype(int)
df_clean['hydrogen_fit'] = df_clean['hydrogen_fit'].astype(int)
df_clean['desalination_fit'] = df_clean['desalination_fit'].astype(int)

# Convert SDG list to string for CSV storage
df_clean['sdg_tags'] = df_clean['sdg_tags'].apply(lambda x: ', '.join(map(str, x)) if x else '')

# Add outreach status column (blank — to be filled by managers)
df_clean['outreach_status'] = 'Not contacted'
df_clean['outreach_notes'] = ''

print(f'Clean dataset shape: {df_clean.shape}')
df_clean[['manager_name', 'country_primary', 'fund_type', 'eif_commitment_eur', 'solar_fit', 'hydrogen_fit', 'desalination_fit']]

## 11. Export Clean CSV

In [ ]:
output_path = '../01_data/processed/eif_fund_managers_clean.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'Clean data exported to: {output_path}')
print(f'Rows: {len(df_clean)} | Columns: {len(df_clean.columns)}')
print('\nColumn list:')
for col in df_clean.columns:
    print(f'  - {col}')

---

## Summary

The raw Excel file has been cleaned and exported as a structured CSV with:
- Standardized column names
- Clean text fields (no line breaks or extra whitespace)
- Extracted numeric EIF commitment amounts
- Classified fund types
- Project fit flags for Solar, Hydrogen, and Desalination
- SDG tags extracted from text
- Outreach status column for manager tracking

**Next step:** Open `02_exploratory_analysis.ipynb` to explore the data and uncover patterns.